In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from langsmith import Client
client = Client()

In [10]:
dataset_name="Evaluation"
#dataset = client.create_dataset(dataset_name)
client.create_example(
    dataset_id = dataset.id,
    examples = [
        {
            "inputs" : {"question" : "What is LangChain?"},
            "outputs" : {"answer" : "A Framework for building LLM Application."}
        }
    ]
)

TypeError: Client.create_example() got an unexpected keyword argument 'examples'. Did you mean 'example_id'?

In [ ]:
import openai
from langsmith import wrappers
openai_client = wrappers.wrap_openai(openai.OpenAI())
eval_instructions = "You are an expert evaluator in evaluating student's answers"
def correctness(input:dict,output:dict,reference:dict)->bool:
    user_content = f"""
    You are grading the following question:
    {input['question']}
    Here is the real answer:
    {reference['answer']}
    You are grading the following answer:
    {output['response']}
    Respond with CORRECT or INCORRECT:
    Grade:
    """
    response = openai_client.chat.completions(
        model = "gpt-4o-mini",
        temperature = 0,
        messages = [
            {"role" : "system","content" : eval_instructions},
            {"role" : "user", "content" : user_content}
        ]
    ).choices[0].message.content
    return response == "CORRECT"


In [11]:
def concision(output:dict,reference:dict)->bool:
    return int(len(output['response']) < 2 * len(reference['answer']))